# Logistic Regression — f4r 기준 25개 block forward ablation

LightGBM 실험과 **같은 25개 case / SKF 5-fold + Group5(SGKF) / 같은 feature builder**를 사용한다.
Logistic Regression은 GPU를 사용하지 않는다.

- 기준선: `A00_baseline = domain + rollup16 + enc3` (`f4r`)
- C 선택: SKF와 Group5 각각의 outer-train 내부 15% validation, 후보 5개
- 각 CV에서 선택한 fold별 C는 해당 CV의 나머지 24개 case에 그대로 재사용
- outer validation은 C·scaler·feature 선택에 사용하지 않음
- case가 끝날 때마다 OOF, test prediction, submission, JSON, console log 저장
- 중단 후 다시 실행하면 완료된 `(CV, case)`는 건너뜀(두 기준선 보정만 재실행)



In [ ]:
from __future__ import annotations

import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = Path("/kaggle/input").is_dir()

REPO_URL = "https://github.com/cancer-classification-ai/onco-ai.git"
# 2026-08-05 현재 develop의 train_linear.py는 빈 파일이다.
# 이 branch가 develop에 merge된 뒤에는 "develop"으로 바꿔도 된다.
PROJECT_BRANCH = "codex/logistic-shared-tabular-pipeline"

# "kagglehub": Kaggle Dataset slug에서 다운로드
# "drive": DRIVE_DATA_DIR에 이미 올려둔 CSV 3개 사용
DATA_SOURCE = "kagglehub"
KAGGLE_DATASET = "hyunwoo11/onco-data-hack"
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/onco-data-hack")
DATA_ROOT = ""  # 아래 데이터 셀이 DATA_SOURCE에 따라 채운다.
MOUNT_GOOGLE_DRIVE = True
INSTALL_CORE_PACKAGES = True

SEED = 42
N_SPLITS = 5
CVS = ("skf", "sgkf")  # sgkf가 fold_group5
TOPK = 500
C_GRID = [0.001, 0.003, 0.01, 0.03, 0.1]
INNER_VALID_FRACTION = 0.15
SOLVER = "lbfgs"
SCALER = "standard"
MAX_ITER = 1000
TOL = 1e-4
TAG = "logreg_block25_group5_v1"

if IN_COLAB:
    WORK_ROOT = Path("/content")
elif IN_KAGGLE:
    WORK_ROOT = Path("/kaggle/working")
else:
    WORK_ROOT = Path.cwd()
KAGGLE_DOWNLOAD_DIR = WORK_ROOT / "onco-data-hack"

REPO_DIR = WORK_ROOT / "onco-ai-logreg"
if Path.cwd().name == "onco-ai" and (Path.cwd() / "scripts").is_dir():
    REPO_DIR = Path.cwd()

if IN_COLAB:
    OUTPUT_ROOT = Path("/content/drive/MyDrive/onco_logreg_block25_group5")
elif IN_KAGGLE:
    OUTPUT_ROOT = Path("/kaggle/working/onco_logreg_block25_group5")
else:
    OUTPUT_ROOT = REPO_DIR / "artifacts" / "logreg_block25_group5"

print("environment :", "Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "local")
print("repo        :", REPO_DIR)
print("output      :", OUTPUT_ROOT)
print("device      : CPU (GPU는 사용하지 않음)")



## 1. 저장소와 패키지 준비

이미 clone되어 있으면 덮어쓰거나 pull하지 않는다. 출력된 commit SHA를 결과
manifest에도 기록한다.



In [ ]:
if IN_COLAB and MOUNT_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)

if not (REPO_DIR / ".git").is_dir():
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            PROJECT_BRANCH,
            "--single-branch",
            REPO_URL,
            str(REPO_DIR),
        ],
        check=True,
    )

if INSTALL_CORE_PACKAGES:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "numpy",
            "pandas",
            "scipy",
            "scikit-learn",
            "pyarrow",
            "PyYAML",
            "kagglehub",
        ],
        check=True,
    )

os.chdir(REPO_DIR)
commit_sha = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], text=True
).strip()
branch_name = subprocess.check_output(
    ["git", "branch", "--show-current"], text=True
).strip()
print("branch      :", branch_name)
print("commit      :", commit_sha)

linear_driver = REPO_DIR / "scripts/train_linear.py"
linear_model = REPO_DIR / "src/cancer_hack/models_linear.py"
if not linear_driver.is_file() or linear_driver.stat().st_size < 1000 or not linear_model.is_file():
    raise RuntimeError(
        "fold-safe Logistic 코드가 없는 branch입니다. "
        f"PROJECT_BRANCH={PROJECT_BRANCH!r}를 사용하거나 해당 branch를 develop에 merge하세요."
    )



## 2. train/test/process 위치 확인

`DATA_SOURCE="kagglehub"`이면 Dataset을 다운로드한 뒤 반환된 실제 경로를 사용한다.
비공개 Dataset은 Colab Secrets에 `KAGGLE_API_TOKEN`이 필요하다. `drive`이면
`DRIVE_DATA_DIR`를 그대로 사용한다.

지원 layout:

- `<DATA_ROOT>/raw` + `<DATA_ROOT>/process`
- `<DATA_ROOT>/data/raw` + `<DATA_ROOT>/data/process`
- `<DATA_ROOT>`에 CSV가 직접 있고 `<DATA_ROOT>/process`에 parquet

process 파일이 Dataset에 있으면 작업 폴더에 symlink하고, 누락분만 다음 셀에서 만든다.



In [ ]:
def valid_raw(path: Path) -> bool:
    return all((path / name).is_file() for name in ("train.csv", "test.csv"))


def resolve_data_layout(explicit: str) -> tuple[Path, Path | None]:
    roots: list[Path] = []
    if explicit:
        roots.append(Path(explicit).expanduser())
    if IN_KAGGLE:
        roots.extend(sorted(Path("/kaggle/input").glob("*")))
        roots.extend(sorted(Path("/kaggle/input").glob("*/*/*")))
    roots.extend(
        [
            REPO_DIR / "data",
            Path("/content/onco-ai/data"),  # 기존 LightGBM Colab clone
            Path("/content/onco-ai"),
            Path("/content/data"),
            Path("/content"),  # Colab 파일 패널에 CSV를 직접 업로드한 경우
            Path("/content/onco-data-hack"),
            Path("/content/drive/MyDrive/onco-data-hack"),
        ]
    )
    checked = []
    for root in roots:
        layouts = [
            (root / "raw", root / "process"),
            (root / "data/raw", root / "data/process"),
            (root, root / "process"),
        ]
        for raw, process in layouts:
            checked.append(str(raw))
            if valid_raw(raw):
                return raw.resolve(), process.resolve() if process.is_dir() else None

    # 폴더 이름이 다른 경우 /content 아래 최대 4단계에서 train.csv를 찾는다.
    discovered = []
    content = Path("/content")
    if content.is_dir():
        for pattern in (
            "train.csv",
            "*/train.csv",
            "*/*/train.csv",
            "*/*/*/train.csv",
            "*/*/*/*/train.csv",
        ):
            discovered.extend(content.glob(pattern))
    for train_path in sorted(set(discovered)):
        raw = train_path.parent
        if not valid_raw(raw):
            continue
        process_candidates = (raw.parent / "process", raw / "process")
        process = next((path for path in process_candidates if path.is_dir()), None)
        return raw.resolve(), process.resolve() if process is not None else None
    raise FileNotFoundError(
        "train.csv/test.csv가 현재 Colab에 없습니다. Drive/업로드 위치를 "
        "DATA_ROOT에 지정하세요.\n확인한 raw 후보:\n"
        + "\n".join(checked[:40])
        + "\n발견한 train.csv:\n"
        + ("\n".join(map(str, discovered)) if discovered else "(없음)")
    )


if DATA_SOURCE == "kagglehub":
    # Colab Secrets(열쇠 아이콘)에 KAGGLE_API_TOKEN을 등록하면 자동 사용한다.
    if IN_COLAB and not os.environ.get("KAGGLE_API_TOKEN"):
        try:
            from google.colab import userdata

            token = userdata.get("KAGGLE_API_TOKEN")
            if token:
                os.environ["KAGGLE_API_TOKEN"] = token
        except Exception:
            pass
    import kagglehub

    try:
        downloaded = Path(
            kagglehub.dataset_download(
                KAGGLE_DATASET,
                output_dir=str(KAGGLE_DOWNLOAD_DIR),
            )
        )
    except Exception as exc:
        raise RuntimeError(
            "Kaggle Dataset 다운로드 실패. Colab Secrets에 KAGGLE_API_TOKEN을 "
            "등록했는지 확인하세요. 403이면 Kaggle Dataset 페이지에서 필요한 "
            "동의/접근 권한도 먼저 완료해야 합니다."
        ) from exc
    DATA_ROOT = str(downloaded)
elif DATA_SOURCE == "drive":
    DATA_ROOT = str(DRIVE_DATA_DIR)
else:
    raise ValueError("DATA_SOURCE는 'kagglehub' 또는 'drive'여야 합니다.")

print("selected data source:", DATA_SOURCE, DATA_ROOT)
RAW_SOURCE, PROCESS_SOURCE = resolve_data_layout(DATA_ROOT)
PROCESS_DIR = REPO_DIR / "data/process"
PROCESS_DIR.mkdir(parents=True, exist_ok=True)

if PROCESS_SOURCE is not None and PROCESS_SOURCE != PROCESS_DIR.resolve():
    for source in PROCESS_SOURCE.iterdir():
        if not source.is_file():
            continue
        target = PROCESS_DIR / source.name
        if not target.exists():
            target.symlink_to(source)

required_raw = ["train.csv", "test.csv", "sample_submission.csv"]
missing_raw = [name for name in required_raw if not (RAW_SOURCE / name).is_file()]
if missing_raw:
    raise FileNotFoundError(f"RAW_SOURCE에 없는 파일: {missing_raw}")

print("raw source    :", RAW_SOURCE)
print("process source:", PROCESS_SOURCE)
print("process work  :", PROCESS_DIR)
print("existing parquet:", len(list(PROCESS_DIR.glob("*.parquet"))))



## 3. 누락된 feature parquet만 생성

`freq21`, `aatrans9`는 누수 방지를 위해 미리 만들지 않고 각 fold-train에서 fit한다.



In [ ]:
FEATURE_COMMANDS = {
    "domain_features.parquet": ["domain"],
    "sample_mutation_features.parquet": ["sample"],
    "sample_mutation_features_rollup.parquet": ["sample", "--include-cell-rollup"],
    "mutation_encoded.parquet": ["enc3"],
    "gene_event_count_matrix.parquet": ["gene", "--kind", "event_count"],
    "gene_mutated_matrix.parquet": ["gene", "--kind", "mutated"],
    "gene_mutation_type_matrix.parquet": ["gene-types"],
    "exact_mutation_tokens.parquet": ["tokens"],
    "signature_mutation_tokens.parquet": ["sigtokens"],
    "parsed_mutation_tokens.parquet": ["parsed-tokens"],
    "mutation_parsed_features.parquet": ["parsed"],
    "additional_burden_features.parquet": ["burden-extra"],
    "amino_acid_features.parquet": ["amino"],
}

for split in ("train", "test"):
    for suffix, command in FEATURE_COMMANDS.items():
        output = PROCESS_DIR / f"{split}_{suffix}"
        if output.is_file():
            continue
        cmd = [
            sys.executable,
            "scripts/make_features.py",
            command[0],
            "--split",
            split,
            "--input",
            str(RAW_SOURCE / f"{split}.csv"),
            "--output",
            str(output),
            *command[1:],
        ]
        print("RUN:", " ".join(cmd))
        subprocess.run(cmd, check=True)

fold_path = PROCESS_DIR / "train_folds.parquet"
if not fold_path.is_file():
    subprocess.run(
        [
            sys.executable,
            "scripts/make_folds.py",
            "--input",
            str(RAW_SOURCE / "train.csv"),
            "--out",
            str(fold_path),
            "--n-splits",
            str(N_SPLITS),
            "--seed",
            str(SEED),
        ],
        check=True,
    )

expected = [
    PROCESS_DIR / f"{split}_{suffix}"
    for split in ("train", "test")
    for suffix in FEATURE_COMMANDS
] + [fold_path]
missing = [str(path) for path in expected if not path.is_file()]
if missing:
    raise FileNotFoundError("생성 후에도 누락된 파일:\n" + "\n".join(missing))
print(f"feature check OK: {len(expected)} files")



## 4. 25개 경우의 수 등록 및 누락 검사

A00~A19는 baseline과 baseline+블록 19개다. E00~E04는
`enc3 × ebovr × ebbnb` 8개 조합 중 A00/A18/A19를 제외한 나머지 5개다.



In [ ]:
import importlib.util
import json
import time
from collections import OrderedDict
from contextlib import redirect_stdout
from datetime import datetime, timezone
from types import SimpleNamespace

import numpy as np
import pandas as pd
import scipy
import sklearn


def load_module(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"cannot import {path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


train_linear = load_module("notebook_train_linear", REPO_DIR / "scripts/train_linear.py")
tg = train_linear.load_tabular_driver()

BASE = ("domain", "rollup16", "enc3")
SINGLE_ADDITIONS = (
    "fe25",
    "rollup",
    "gec",
    "gtype",
    "parsed19",
    "burden8",
    "aa9",
    "sigtok",
    "exacttok",
    "ptok",
    "freq21",
    "aatrans9",
    "comut",
    "lsvd",
    "lnmf",
    "gmod",
    "csig",
    "ebovr",
    "ebbnb",
)

CASES = OrderedDict()
CASES["A00_baseline"] = BASE
for number, block in enumerate(SINGLE_ADDITIONS, start=1):
    CASES[f"A{number:02d}_plus_{block}"] = (*BASE, block)
CASES.update(
    {
        "E00_no_enc3": ("domain", "rollup16"),
        "E01_ebovr_only": ("domain", "rollup16", "ebovr"),
        "E02_ebbnb_only": ("domain", "rollup16", "ebbnb"),
        "E03_ebboth_no_enc3": ("domain", "rollup16", "ebovr", "ebbnb"),
        "E04_enc3_ebboth": ("domain", "rollup16", "enc3", "ebovr", "ebbnb"),
    }
)

for name, blocks in CASES.items():
    tg.CONFIGS[name] = {
        "blocks": tuple(blocks),
        "weight": "balanced",
        "desc": f"LogReg block25: {' + '.join(blocks)}",
    }

registered_blocks = set(tg.BLOCK_DESC)
covered_blocks = set(BASE) | set(SINGLE_ADDITIONS)
assert len(CASES) == 25, len(CASES)
assert len(SINGLE_ADDITIONS) == 19
assert registered_blocks == covered_blocks, {
    "missing": sorted(registered_blocks - covered_blocks),
    "unknown": sorted(covered_blocks - registered_blocks),
}
factorial = {
    tuple(block for block in blocks if block in {"enc3", "ebovr", "ebbnb"})
    for blocks in CASES.values()
}
assert len(factorial) == 8, factorial

case_table = pd.DataFrame(
    [
        {
            "case": name,
            "blocks": " + ".join(blocks),
            "added_block": (
                "baseline"
                if name == "A00_baseline"
                else name.split("_plus_", 1)[1]
                if "_plus_" in name
                else "factorial"
            ),
        }
        for name, blocks in CASES.items()
    ]
)
display(case_table)
print("25 cases / registered 22 blocks coverage OK")



## 5. 공용 Dataset 로드와 실행 설정

이 셀은 모든 block을 한 번만 메모리에 올린다. `DRY_RUN=True`로 먼저 검사만 할 수 있다.



In [ ]:
DRY_RUN = False
RUN_SUBMISSION = True
RESUME_COMPLETED_CASES = True

ARTIFACTS = OUTPUT_ROOT / "artifacts"
STATE_PATH = OUTPUT_ROOT / "suite_state.json"
FAILURE_PATH = OUTPUT_ROOT / "failed_cases.json"
REPORT_DIR = OUTPUT_ROOT / "reports"
CONSOLE_DIR = ARTIFACTS / "console"
for directory in (ARTIFACTS, REPORT_DIR, CONSOLE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

tg.RAW_DIR = RAW_SOURCE
tg.PROC_DIR = PROCESS_DIR
tg.ARTIFACTS = ARTIFACTS

all_needed = set().union(*(set(blocks) for blocks in CASES.values()))
data = tg.Dataset(all_needed, n_splits=N_SPLITS)

linear_config = train_linear.load_yaml(REPO_DIR / "configs/linear.yaml")
notebook_cli = [
    "--configs",
    ",".join(CASES),
    "--cv",
    "all",
    "--n-splits",
    str(N_SPLITS),
    "--seed",
    str(SEED),
    "--tag",
    TAG,
    "--artifacts",
    str(ARTIFACTS),
    "--raw-dir",
    str(RAW_SOURCE),
    "--process-dir",
    str(PROCESS_DIR),
    "--calibration-config",
    "A00_baseline",
    "--c-grid",
    ",".join(map(str, C_GRID)),
    "--inner-valid-fraction",
    str(INNER_VALID_FRACTION),
    "--solver",
    SOLVER,
    "--scaler",
    SCALER,
    "--max-iter",
    str(MAX_ITER),
    "--tol",
    str(TOL),
    "--submission" if RUN_SUBMISSION else "--no-submission",
]
linear_args, forwarded = train_linear.build_parser(linear_config).parse_known_args(
    notebook_cli
)
driver_args = train_linear.driver_args(tg, linear_args, forwarded)
driver_args.topk = TOPK
driver_args.dry_run = DRY_RUN

trainer = train_linear.LinearFoldTrainer(
    calibration_config="A00_baseline",
    c_grid=C_GRID,
    inner_valid_fraction=INNER_VALID_FRACTION,
    solver=SOLVER,
    scaler=SCALER,
    max_iter=MAX_ITER,
    tol=TOL,
    seed=SEED,
    n_splits=N_SPLITS,
)

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "git_branch": branch_name,
    "git_commit": commit_sha,
    "raw_source": str(RAW_SOURCE),
    "process_dir": str(PROCESS_DIR),
    "output_root": str(OUTPUT_ROOT),
    "python": platform.python_version(),
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "scikit_learn": sklearn.__version__,
    "cv": list(CVS),
    "n_splits": N_SPLITS,
    "seed": SEED,
    "topk": TOPK,
    "model": {
        "solver": SOLVER,
        "scaler": SCALER,
        "max_iter": MAX_ITER,
        "tol": TOL,
    },
    "calibration": {
        "config": "A00_baseline",
        "c_grid": C_GRID,
        "inner_valid_fraction": INNER_VALID_FRACTION,
        "outer_validation_used": False,
    },
    "cases": {name: list(blocks) for name, blocks in CASES.items()},
}
train_linear.atomic_json(OUTPUT_ROOT / "manifest.json", manifest)
print("Dataset/config ready")



## 6. 기준선 보정 실행

SKF와 Group5에서 fold별 C를 각각 고른다. 재시작할 때도 공정한 재사용을 위해
두 기준선은 다시 돈다. SKF 기준 로컬 실측은 약 45초였다.



In [ ]:
class Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, value):
        for stream in self.streams:
            stream.write(value)
        return len(value)

    def flush(self):
        for stream in self.streams:
            stream.flush()


def load_json(path: Path, default):
    if not path.is_file():
        return default
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)


def result_marker(result: dict) -> dict:
    stem = result["stem"]
    return {
        "cv": result["cv"],
        "case": result["config"],
        "stem": stem,
        "blocks": result["blocks"],
        "n_features_per_fold": result["n_features_per_fold"],
        "oof_macro_f1": result["oof_macro_f1"],
        "oof_macro_f1_singleton": result["oof_macro_f1_singleton"],
        "oof_accuracy": result["oof_accuracy"],
        "fold_macro_f1": result["fold_macro_f1"],
        "elapsed_seconds": result["elapsed_seconds"],
        "per_class_f1": result["per_class_f1"],
        "fold_models": result["fold_models"],
        "log_path": str(ARTIFACTS / "logs" / f"{stem}.json"),
        "oof_path": str(ARTIFACTS / "oof" / f"oof_{stem}.csv"),
        "test_path": str(ARTIFACTS / "test_predictions" / f"test_{stem}.csv"),
        "submission_path": str(
            ARTIFACTS / "submissions" / f"submission_{stem}.csv"
        ),
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    }


def state_key(cv: str, case_name: str) -> str:
    return f"{cv}::{case_name}"


def save_result(result: dict, state: dict, cv: str) -> None:
    result["linear_calibration"] = trainer.summary(cv)
    result["runtime_versions"] = {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "scipy": scipy.__version__,
        "scikit_learn": sklearn.__version__,
    }
    train_linear.atomic_json(
        ARTIFACTS / "logs" / f"{result['stem']}.json",
        result,
    )
    state[state_key(cv, result["config"])] = result_marker(result)
    train_linear.atomic_json(STATE_PATH, state)


state = load_json(STATE_PATH, {})
baseline_name = "A00_baseline"
baseline_results = {}
for cv in CVS:
    print(f"\n[CALIBRATION] {cv} / {baseline_name}")
    trainer.begin_config(config=baseline_name, cv=cv)
    baseline_console = CONSOLE_DIR / f"{cv}_{baseline_name}.log"
    with baseline_console.open("w", encoding="utf-8") as log_handle:
        with redirect_stdout(Tee(sys.stdout, log_handle)):
            baseline_result = tg.run_config(
                data,
                config=baseline_name,
                cv=cv,
                args=driver_args,
                fit_model=trainer,
            )
    save_result(baseline_result, state, cv)
    baseline_results[cv] = baseline_result
    print(f"{cv} selected C per fold:", trainer.selected_c[cv])



## 7. SKF·Group5 각각 나머지 24개 case 실행

완료 marker와 OOF/test/log 파일이 모두 있을 때만 skip한다. 실패한 case는
`failed_cases.json`에 남기고 다음 case를 계속 실행한다.



In [ ]:
def marker_is_complete(marker: dict) -> bool:
    required = ["log_path", "oof_path", "test_path"]
    if RUN_SUBMISSION:
        required.append("submission_path")
    return all(Path(marker.get(key, "")).is_file() for key in required)


failures = load_json(FAILURE_PATH, {})
remaining = list(CASES)[1:]
suite_started = time.perf_counter()
total_runs = len(CVS) * len(CASES)
run_index = len(CVS)  # 앞 셀에서 실행한 CV별 baseline 수

for cv in CVS:
    for case_name in remaining:
        run_index += 1
        key = state_key(cv, case_name)
        existing = state.get(key)
        if (
            RESUME_COMPLETED_CASES
            and existing is not None
            and marker_is_complete(existing)
        ):
            print(f"[{run_index:02d}/{total_runs}] SKIP {cv} / {case_name}")
            continue

        print(f"\n[{run_index:02d}/{total_runs}] START {cv} / {case_name}")
        trainer.begin_config(config=case_name, cv=cv)
        console_path = CONSOLE_DIR / f"{cv}_{case_name}.log"
        failure_key = state_key(cv, case_name)
        try:
            with console_path.open("w", encoding="utf-8") as log_handle:
                with redirect_stdout(Tee(sys.stdout, log_handle)):
                    result = tg.run_config(
                        data,
                        config=case_name,
                        cv=cv,
                        args=driver_args,
                        fit_model=trainer,
                    )
            save_result(result, state, cv)
            failures.pop(failure_key, None)
            train_linear.atomic_json(FAILURE_PATH, failures)
            delta = (
                result["oof_macro_f1"]
                - baseline_results[cv]["oof_macro_f1"]
            )
            print(
                f"[{run_index:02d}/{total_runs}] DONE {cv} / {case_name}: "
                f"F1={result['oof_macro_f1']:.6f}, delta={delta:+.6f}"
            )
        except Exception as exc:
            failures[failure_key] = {
                "cv": cv,
                "case": case_name,
                "type": type(exc).__name__,
                "message": str(exc),
                "console_path": str(console_path),
                "failed_at_utc": datetime.now(timezone.utc).isoformat(),
            }
            train_linear.atomic_json(FAILURE_PATH, failures)
            print(
                f"[{run_index:02d}/{total_runs}] FAILED {cv} / {case_name}: "
                f"{type(exc).__name__}: {exc}"
            )

print(f"suite cell elapsed: {(time.perf_counter() - suite_started) / 3600:.2f} h")
print(f"completed: {len(state)}/{total_runs}, failed: {len(failures)}")



## 8. 비교표와 그래프

기준선 대비 delta가 양수면 초록색, 음수면 빨간색으로 표시한다.



In [ ]:
state = load_json(STATE_PATH, {})
missing_baselines = [
    cv for cv in CVS if state_key(cv, baseline_name) not in state
]
if missing_baselines:
    raise RuntimeError(f"baseline result가 없는 CV: {missing_baselines}")

rows = []
for cv in CVS:
    baseline_f1 = state[state_key(cv, baseline_name)]["oof_macro_f1"]
    for case_name in CASES:
        marker = state.get(state_key(cv, case_name))
        if marker is None:
            continue
        fold_models = marker.get("fold_models", [])
        converged = sum(bool(model.get("converged")) for model in fold_models)
        rows.append(
            {
                "cv": "SKF" if cv == "skf" else "Group5",
                "case": case_name,
                "added_block": case_table.set_index("case").loc[case_name, "added_block"],
                "features_mean": np.mean(marker["n_features_per_fold"]),
                "oof_macro_f1": marker["oof_macro_f1"],
                "delta_vs_baseline": marker["oof_macro_f1"] - baseline_f1,
                "singleton_f1": marker["oof_macro_f1_singleton"],
                "accuracy": marker["oof_accuracy"],
                "fold_mean": np.mean(marker["fold_macro_f1"]),
                "fold_std": np.std(marker["fold_macro_f1"]),
                "converged_folds": f"{converged}/{N_SPLITS}",
                "elapsed_min": marker["elapsed_seconds"] / 60,
            }
        )

summary = pd.DataFrame(rows).sort_values(
    ["cv", "oof_macro_f1", "case"], ascending=[True, False, True]
)
summary.to_csv(REPORT_DIR / "logreg_block25_summary.csv", index=False)
summary.to_json(
    REPORT_DIR / "logreg_block25_summary.json",
    orient="records",
    force_ascii=False,
    indent=2,
)


def color_delta(value):
    if value > 0:
        return "color: #137333; background-color: #e6f4ea"
    if value < 0:
        return "color: #c5221f; background-color: #fce8e6"
    return "font-weight: bold"


styled = (
    summary.style.format(
        {
            "features_mean": "{:,.0f}",
            "oof_macro_f1": "{:.6f}",
            "delta_vs_baseline": "{:+.6f}",
            "singleton_f1": "{:.6f}",
            "accuracy": "{:.6f}",
            "fold_mean": "{:.6f}",
            "fold_std": "{:.6f}",
            "elapsed_min": "{:.1f}",
        }
    )
    .map(color_delta, subset=["delta_vs_baseline"])
    .hide(axis="index")
)
display(styled)
(REPORT_DIR / "logreg_block25_summary.html").write_text(
    styled.to_html(), encoding="utf-8"
)

per_class_rows = []
for cv in CVS:
    baseline_per_class = state[state_key(cv, baseline_name)]["per_class_f1"]
    for case_name in CASES:
        marker = state.get(state_key(cv, case_name))
        if marker is None:
            continue
        for subclass, score in marker["per_class_f1"].items():
            per_class_rows.append(
                {
                    "cv": "SKF" if cv == "skf" else "Group5",
                    "case": case_name,
                    "SUBCLASS": subclass,
                    "f1": score,
                    "delta_vs_baseline": score - baseline_per_class[subclass],
                }
            )
pd.DataFrame(per_class_rows).to_csv(
    REPORT_DIR / "per_class_f1_delta.csv", index=False
)

try:
    import matplotlib.pyplot as plt

    for cv_label in ("SKF", "Group5"):
        plot_frame = summary[summary["cv"] == cv_label].sort_values(
            "delta_vs_baseline"
        )
        colors = [
            "#137333" if value >= 0 else "#c5221f"
            for value in plot_frame["delta_vs_baseline"]
        ]
        axis = plot_frame.plot.barh(
            x="case",
            y="delta_vs_baseline",
            color=colors,
            legend=False,
            figsize=(10, 9),
        )
        axis.axvline(0, color="#444", linewidth=1)
        axis.set_title(f"Logistic Regression {cv_label}: delta vs A00")
        axis.set_xlabel("Macro F1 delta")
        plt.tight_layout()
        figure_path = REPORT_DIR / f"logreg_block25_delta_{cv_label.lower()}.png"
        plt.savefig(figure_path, dpi=160, bbox_inches="tight")
        plt.show()
except ImportError:
    print("matplotlib 없음: 표는 정상 저장됐고 그래프만 생략합니다.")

print("reports:", REPORT_DIR)



## 9. 전체 산출물 ZIP



In [ ]:
archive_base = WORK_ROOT / f"{TAG}_{commit_sha[:8]}"
archive_path = Path(
    shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_ROOT)
)
print("archive:", archive_path)
print("size MB:", archive_path.stat().st_size / 1024**2)

if IN_COLAB:
    from google.colab import files

    print("필요하면 다음 줄의 주석을 풀어 다운로드하세요.")
    print(f"# files.download({str(archive_path)!r})")
